# EDSR ×3 — 학습부터 적용까지

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).
학습은 코드가 도는지 확인하는 수준으로만 돌리고, 결과는 미리 학습해둔 가중치로 본다.

**런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 켜세요.

## 1. 데이터

In [ ]:
import urllib.request
LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'sr_models.py', 'sr_losses.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)

from sr_utils import *

show([(f'training {i+1}', *pair('training', s)) for i, s in enumerate(REPS['training'])])
show_data()        # validation 2패치 + test 2구역

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 16장으로 1 epoch 만 돌린다.
아래 결과는 전체 데이터로 학습해둔 가중치를 쓴다.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from sr_models import EDSR
from sr_losses import get_loss

N_TRAIN, EPOCHS, BATCH = 16, 1, 4          # 동작 확인용이라 작게 잡았다
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

stems = list_split('training')[:N_TRAIN]
lo, hi = zip(*[pair('training', s) for s in stems])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float()
loader = DataLoader(TensorDataset(to_t(lo), to_t(hi)), batch_size=BATCH, shuffle=True)

net = EDSR().to(dev).train()
opt = torch.optim.Adam(net.parameters(), 1e-4)
crit = get_loss('l1')

for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for x, y in loader:
        loss = crit(net(x.to(dev)), y.to(dev))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(x)
    print(f'epoch {ep}/{EPOCHS}   loss {tot / len(loader.dataset):.3f}')

print(f'\n{N_TRAIN}장으로 {EPOCHS}회. 실제 성능은 아래 미리 학습된 가중치로 확인한다.')

## 3. 결과

In [ ]:
from sr_models import load_edsr

net = load_edsr(fetch(f'{BASE}/models/03_edsr_x3/checkpoints/edsr_x3.pt', 'edsr_x3.pt'))

@torch.no_grad()
def upscale(lr):
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(net.parameters()).device)
    return net(t).clamp(0, 255).round()[0].cpu().numpy().transpose(1, 2, 0).astype('uint8')

print('EDSR x3 준비 완료')

show_results(upscale, 'EDSR', center=[(137, 236), (313, 346)])

## 4. 평가

In [ ]:
rows = compare(upscale, label='EDSR')

## 5. 최종 테스트 — 인천

정답이 없는 실제 Sentinel-2 촬영본이다. 점수는 못 내고 눈으로 확인한다.

In [ ]:
show_test(upscale, 'EDSR', center=(800, 800), size=70)